# ZA Renewable Profile Validation

This notebook checks the South Africa 2023 atlite cutout and renewable profile artifacts produced for Calibration Plan module 03.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import xarray as xr
from IPython.display import display

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "data/za_audit").exists())
VALIDATION = ROOT / "data/za_audit/za_atlite_renewable_profile_validation.csv"
TECHNICAL = ROOT / "data/za_audit/za_atlite_technical_potential.csv"
PROFILE_DIR = ROOT / "resources/za_2023_fixed_validation/renewable_profiles"

plt.rcParams.update({"figure.figsize": (10, 4), "axes.grid": True})

validation = pd.read_csv(VALIDATION)
technical = pd.read_csv(TECHNICAL)
status_counts = validation["status"].value_counts().rename_axis("status").reset_index(name="checks")
display(status_counts)

## Gate A Checks

The validation table records cutout coverage, carrier profile bounds, technical potential diagnostics, and the GEGIS `2023_custom` preflight.

In [ ]:
display(validation)

issues = validation.loc[validation["status"].isin(["warn", "fail"])].copy()
display(issues if not issues.empty else pd.DataFrame({"message": ["No warnings or failures"]}))

## Technical Potential

The values below are diagnostics only. No correction factors or profile scaling are applied in module 03.

In [ ]:
cols = [
    "carrier",
    "hours",
    "p_nom_max_mw",
    "technical_potential_twh",
    "full_load_hours",
    "area_km2",
    "installable_power_density_mw_per_km2",
    "sanity_status",
]
display(technical[cols])

ax = technical.set_index("carrier")["full_load_hours"].dropna().plot(kind="bar", color=["#357ABD", "#4F8A10", "#B26A00"])
ax.set_ylabel("Full-load hours")
ax.set_title("Weighted full-load hours by carrier")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## Daily Availability Shapes

Solar, onwind, and CSP profiles are aggregated to p_nom-weighted daily availability. Hydro is omitted here because the upstream profile is empty until module 08 supplies reconciled ZA hydro plants.

In [ ]:
def weighted_profile(path: Path) -> pd.Series:
    ds = xr.open_dataset(path)
    weights = ds["p_nom_max"].where(ds["p_nom_max"] > 0, 0)
    profile = ds["profile"].weighted(weights).mean("bus").to_series()
    ds.close()
    profile.index = pd.to_datetime(profile.index)
    return profile

series = {
    carrier: weighted_profile(PROFILE_DIR / f"profile_{carrier}.nc").resample("D").mean()
    for carrier in ["solar", "onwind", "csp"]
}
daily = pd.DataFrame(series)
ax = daily.plot(linewidth=1.2)
ax.set_ylabel("Availability p.u.")
ax.set_title("Daily mean renewable availability, 2023")
plt.tight_layout()
plt.show()

display(daily.describe().T)

## CSP Carrier Check

CSP is validated as its own `csp` carrier from `profile_csp.nc`; it is not combined with PV.

In [ ]:
csp_rows = validation.loc[validation["carrier"].eq("csp") | validation["notes"].str.contains("csp|CSP", case=False, na=False)]
display(csp_rows)

## Conclusion

Module 03 Gate A passes for the verified ERA5 cutout and native solar, onwind, and CSP profiles. Hydro is present as an upstream output file but empty because the current pre-module-08 powerplant data has no known ZA hydro plants; the warning is carried forward without scaling or fallback.